# Approach

We trained our base model, which only classifies plant genera, on PlantNet300k. However, plant conditions and health are not examined or included in its annotations. A new problem arises: where can we find specific datasets that mirror PN300k’s classes while assigning specific diseases to each? This is a tall order; to my knowledge, there is no freely available dataset with a suitable license that contains all the necessary classes. However, I found a similar, smaller dataset (PlantVillage) which contains 38 classes across 14 genera. Since each disease is specific to a genus, it is better not to assume specific diseases for the entire PN300k dataset.

In [15]:
import os
from pathlib import Path

if os.path.abspath(os.curdir)[-2:] != "ai":
    os.chdir("..")

data_dir = Path(os.path.abspath(os.curdir) + "/data/plant_village_data/color")
subfolders = [f.name for f in data_dir.iterdir() if f.is_dir()]
print(subfolders)

['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Sp

```mermaid
flowchart TD
    A[Input Image \224x224] --> B[ResNet50 backbone]
    B --> C[Shared bottleneck]

    C --> D[Head: class/genus]
    C --> E[Head: healthy/sick]
    C --> F[Head: disease]

    E --> G{Is sick?}
    
    G -- NO --> H[healthy + conf.]
    G -- YES --> I[disease name + conf.]
```

# Model

In [21]:
print (Path(os.path.abspath(os.curdir)))

from torchinfo import summary
from src.plant_care_ai.models.multitask_resnet import MultitaskResNet

_mt = MultitaskResNet()
summary(_mt, input_size=(1, 3, 224, 224))

/home/szczuru/projects/Plant-Care-Assistant-App/ai


Layer (type:depth-idx)                        Output Shape              Param #
MultitaskResNet                               [1, 38]                   --
├─Sequential: 1-1                             [1, 2048, 1, 1]           --
│    └─Conv2d: 2-1                            [1, 64, 112, 112]         9,408
│    └─BatchNorm2d: 2-2                       [1, 64, 112, 112]         128
│    └─ReLU: 2-3                              [1, 64, 112, 112]         --
│    └─MaxPool2d: 2-4                         [1, 64, 56, 56]           --
│    └─Sequential: 2-5                        [1, 256, 56, 56]          --
│    │    └─Bottleneck: 3-1                   [1, 256, 56, 56]          75,008
│    │    └─Bottleneck: 3-2                   [1, 256, 56, 56]          70,400
│    │    └─Bottleneck: 3-3                   [1, 256, 56, 56]          70,400
│    └─Sequential: 2-6                        [1, 512, 28, 28]          --
│    │    └─Bottleneck: 3-4                   [1, 512, 28, 28]          379,392

Temporarily, only resnet50 model ()